# Hands-on Lab: Interactive Visual Analytics with Folium

*JupyterLite (Pyodide) version.*

This lab has 3 tasks:

* **TASK 1:** Mark all launch sites on a map
* **TASK 2:** Mark the success/failed launches for each site on the map
* **TASK 3:** Calculate the distances between a launch site and its proximities

## Setup

In [ ]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])


In [ ]:
import folium
import pandas as pd

# folium plugins
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon


## Load the data

In [ ]:
# JupyterLite runs Python in the browser (Pyodide), so pandas can't open a network
# socket directly (pd.read_csv(URL) will fail/hang). Instead we fetch the file with the
# browser's own fetch API and hand the bytes to pandas.
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df = pd.read_csv(spacex_csv_file)

# Keep only the columns we need: site name, coordinates, and launch outcome
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]

# One row per unique launch site, to get each site's coordinates
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]

launch_sites_df


## TASK 1: Mark all launch sites on a map

In [ ]:
# Start location is NASA Johnson Space Center (Houston, TX) -- used as the map's initial view
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# For each launch site, add a Circle (highlighted area) and a text label Marker
for index, site in launch_sites_df.iterrows():
    coordinate = [site['Lat'], site['Long']]
    site_name = site['Launch Site']

    circle = folium.Circle(
        coordinate,
        radius=1000,
        color='#d35400',
        fill=True
    ).add_child(folium.Popup(site_name))

    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % site_name,
        )
    )

    site_map.add_child(circle)
    site_map.add_child(marker)

site_map


Explore the map: are all launch sites in proximity to the Equator line? Are they close to the coast? (Yes -- rockets launch eastward, near the equator for an extra speed boost, and near the coast so spent stages fall over open ocean.)

## TASK 2: Mark the success/failed launches for each site

In [ ]:
spacex_df.tail(10)


Create a `marker_color` column: green for a successful launch (`class == 1`), red for a failed launch (`class == 0`).

In [ ]:
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'

spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)


Many launches share the same site (and therefore the same coordinate), so we use a `MarkerCluster` to keep the map readable, then add one colored marker per launch record.

In [ ]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

for index, record in spacex_df.iterrows():
    coordinate = [record['Lat'], record['Long']]
    marker = folium.Marker(
        coordinate,
        icon=folium.Icon(color='white', icon_color=record['marker_color'])
    )
    marker_cluster.add_child(marker)

site_map


From the color-coded clusters you can already see which sites have relatively higher success rates.

## TASK 3: Calculate the distances between a launch site and its proximities

In [ ]:
# Add MousePosition so hovering over the map shows the (Lat, Long) under the cursor.
# Use this to read off the coordinates of any coastline/city/railway/highway point you want to measure.
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map


In [ ]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance


**Coastline distance.** Hover on the map near CCAFS SLC-40 with `MousePosition` to find the closest coastline point, then plug the coordinates in below. The values here are a worked example -- replace them with whatever point you find on your own map.

In [ ]:
# Example launch site: CCAFS SLC-40
launch_site_lat = 28.563197
launch_site_lon = -80.576820

# Example closest coastline point (replace with what you read off MousePosition)
coastline_lat = 28.56367
coastline_lon = -80.57163

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f'Distance from CCAFS SLC-40 to the coastline point: {distance_coastline:.2f} KM')


In [ ]:
# Add a marker at the coastline point, labeled with the computed distance
coordinate = [coastline_lat, coastline_lon]
distance_marker = folium.Marker(
    coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
    )
)
site_map.add_child(distance_marker)
site_map


**Draw a line** between the launch site and the coastline point.

In [ ]:
coordinates = [[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]]
lines = folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)
site_map


**Closest city / railway / highway.** Same idea -- find a point with `MousePosition`, compute the distance, and draw a marker + line. Here's a reusable helper plus one worked example (a nearby railway); repeat the call for a highway or city point of your own.

In [ ]:
def add_proximity_marker(site_map, launch_site_lat, launch_site_lon, point_lat, point_lon, label):
    """Add a distance-labeled marker + connecting line from a launch site to a nearby point of interest."""
    distance = calculate_distance(launch_site_lat, launch_site_lon, point_lat, point_lon)

    marker = folium.Marker(
        [point_lat, point_lon],
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s: %s</b></div>' % (
                label, "{:10.2f} KM".format(distance)
            ),
        )
    )
    site_map.add_child(marker)

    line = folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], [point_lat, point_lon]],
        weight=1
    )
    site_map.add_child(line)

    return distance

# Worked example: nearby railway point (replace coordinates with your own MousePosition reading)
railway_lat, railway_lon = 28.5721, -80.5853
railway_distance = add_proximity_marker(
    site_map, launch_site_lat, launch_site_lon, railway_lat, railway_lon, "Railway"
)
print(f'Distance to railway: {railway_distance:.2f} KM')

site_map


Now that distance lines are on the map, you can answer:

* Are launch sites close to railways?
* Are launch sites close to highways?
* Are launch sites close to the coastline?
* Do launch sites keep a distance from cities?

(Typically: yes to railways/highways/coastline -- for transporting rocket stages and for safety in case of a launch failure -- and yes, sites are kept well away from cities for public safety.)

## Next Steps

With these interactive maps built, the next step is a dashboard (e.g. with Plotly Dash) on the detailed launch records.